# Embryo Stage Classification

This notebook trains four pretrained CNN backbones on the Kaggle embryo dataset:
- MobileNetV2
- VGG16
- VGG19
- InceptionV3

Each model is trained twice: once with weighted Cross Entropy (CE) and once with a custom Smooth Ordinal Loss that penalises stage-distance errors exponentially. Results from all eight experiments are compared at the end.

In [1]:
import torch

print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))

True
12.8
Tesla T4


In [2]:
import os
import copy
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.models import (
    mobilenet_v2,
    vgg16,
    vgg19,
    inception_v3,
    MobileNet_V2_Weights,
    VGG16_Weights,
    VGG19_Weights,
    Inception_V3_Weights,
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

Using device: cuda


## Configuration and Kaggle Dataset Paths

In [3]:
# -----------------------------------------------------------------------
# Kaggle dataset paths
# Add the dataset via:  Data -> Add Data -> search "embryo dataset"
# Dataset by abhishekbuddiga06 on Kaggle
# -----------------------------------------------------------------------
DATA_DIR = '/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset/embryo_dataset'
ANN_DIR = '/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset_annotations/embryo_dataset_annotations'
PHASES = [
    'tPB2', 'tPNa', 'tPNf', 't2', 't3', 't4', 't5', 't6',
    't7', 't8', 't9+', 'tM', 'tSB', 'tB', 'tEB', 'tHB'
]
label_map = {phase: idx for idx, phase in enumerate(PHASES)}

@dataclass
class Config:
    sample_rate: int = 8
    epochs: int = 10
    batch_size: int = 32
    inception_batch_size: int = 16
    image_size: int = 224
    inception_size: int = 299
    lr: float = 1e-4
    weight_decay: float = 1e-4
    num_workers: int = 2
    random_state: int = 42
    label_smoothing: float = 0.1
    grad_clip: float = 1.0
    # SmoothOrdinalLoss hyperparameters
    sol_alpha: float = 0.3
    sol_tau: float = 3.0
    aux_weight: float = 0.4
    merge_source: int = 15
    merge_target: int = 14

cfg = Config()
print(cfg)

Config(sample_rate=8, epochs=10, batch_size=32, inception_batch_size=16, image_size=224, inception_size=299, lr=0.0001, weight_decay=0.0001, num_workers=2, random_state=42, label_smoothing=0.1, grad_clip=1.0, sol_alpha=0.3, sol_tau=3.0, aux_weight=0.4, merge_source=15, merge_target=14)


In [4]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(cfg.random_state)

## Build Sampled Dataframe

Annotation CSV files map each embryo stage to a [start, end] frame range.
One frame is sampled every `sample_rate` frames within each annotated interval to reduce redundancy while preserving stage progression information.

In [5]:
import os

ANN_DIR = '/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset_annotations/embryo_dataset_annotations'
print(os.listdir(ANN_DIR)[:5])

['BC750-7_phases.csv', 'SK308-7_phases.csv', 'QC697-4_phases.csv', 'DI358-3_phases.csv', 'DV210-4_phases.csv']


In [6]:
def build_dataframe(sample_rate: int = 8) -> pd.DataFrame:
    rows = []

    annotation_files = sorted([f for f in os.listdir(ANN_DIR) if f.endswith('.csv')])

    for file_name in tqdm(annotation_files, desc='Reading annotations'):
        embryo_id = file_name.replace('_phases.csv', '')
        ann_path  = os.path.join(ANN_DIR, file_name)
        image_dir = os.path.join(DATA_DIR, embryo_id)

        if not os.path.isdir(image_dir):
            continue

        image_files = sorted(os.listdir(image_dir))
        if not image_files:
            continue

        ann_df = pd.read_csv(ann_path, header=None, names=['phase', 'start', 'end'])

        for _, row in ann_df.iterrows():
            phase = row['phase']
            if phase not in label_map:
                continue

            start = int(row['start'])
            end   = int(row['end'])
            label = label_map[phase]

            for frame_idx in range(start, end, sample_rate):
                if 0 <= frame_idx < len(image_files):
                    rows.append({
                        'image':  os.path.join(image_dir, image_files[frame_idx]),
                        'label':  label,
                        'embryo': embryo_id,
                    })

    return pd.DataFrame(rows)

df = build_dataframe(cfg.sample_rate)
print('Total samples:', len(df))
display(df.head())

Reading annotations:   0%|          | 0/704 [00:00<?, ?it/s]

Total samples: 40337


,image,label,embryo
0,/kaggle/input/datasets/abhishekbuddiga06/embry...,0,AA83-7
1,/kaggle/input/datasets/abhishekbuddiga06/embry...,0,AA83-7
2,/kaggle/input/datasets/abhishekbuddiga06/embry...,0,AA83-7
3,/kaggle/input/datasets/abhishekbuddiga06/embry...,1,AA83-7
4,/kaggle/input/datasets/abhishekbuddiga06/embry...,1,AA83-7


In [7]:
# Split embryo-wise so no embryo appears in more than one split.
# This avoids data leakage through temporally correlated frames.
embryos = df['embryo'].unique()

train_ids, temp_ids = train_test_split(
    embryos, test_size=0.30, random_state=cfg.random_state, shuffle=True
)
val_ids, test_ids = train_test_split(
    temp_ids, test_size=0.50, random_state=cfg.random_state, shuffle=True
)

train_df = df[df['embryo'].isin(train_ids)].copy()
val_df   = df[df['embryo'].isin(val_ids)].copy()
test_df  = df[df['embryo'].isin(test_ids)].copy()

print('Split sizes:')
print('  Train:', len(train_df))
print('  Val  :', len(val_df))
print('  Test :', len(test_df))

Split sizes:
  Train: 28123
  Val  : 6156
  Test : 6058


In [8]:
# Class 15 (tHB) is merged into class 14 (tEB) because it has very few
# annotated samples and is visually the most similar to the preceding stage.
print('Train label counts before merge:')
display(train_df['label'].value_counts().sort_index())

for split_df in [train_df, val_df, test_df]:
    split_df.loc[split_df['label'] == cfg.merge_source, 'label'] = cfg.merge_target

num_classes = len(PHASES) - 1  # 15 effective classes

print('Train label counts after merge:')
display(train_df['label'].value_counts().sort_index())
print('Effective num_classes:', num_classes)

Train label counts before merge:


label
0      911
1     3907
2      770
3     2716
4      598
5     2714
6      862
7      898
8     1016
9     2990
10    4671
11    1620
12    1579
13     990
14    1877
15       4
Name: count, dtype: int64

Train label counts after merge:


label
0      911
1     3907
2      770
3     2716
4      598
5     2714
6      862
7      898
8     1016
9     2990
10    4671
11    1620
12    1579
13     990
14    1881
Name: count, dtype: int64

Effective num_classes: 15


In [9]:
# Compute inverse-frequency class weights for the CE baseline.
classes      = np.arange(num_classes)
weights      = compute_class_weight(class_weight='balanced', classes=classes, y=train_df['label'].values)
class_weights = torch.tensor(weights, dtype=torch.float32, device=DEVICE)

weight_table = pd.DataFrame({'class_id': classes, 'weight': weights})
display(weight_table)

,class_id,weight
0,0,2.058031
1,1,0.479874
2,2,2.434892
3,3,0.690304
4,4,3.135229
5,5,0.690813
6,6,2.175019
7,7,2.087825
8,8,1.845341
9,9,0.627046


## Dataset and Dataloaders

In [10]:
class EmbryoDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform=None):
        self.frame     = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        for _ in range(5):
            row = self.frame.iloc[idx]
            try:
                image = Image.open(row['image']).convert('RGB')
                if self.transform is not None:
                    image = self.transform(image)
                label = torch.tensor(int(row['label']), dtype=torch.long)
                return image, label
            except Exception:
                idx = random.randint(0, len(self.frame) - 1)

        fallback = torch.zeros(3, cfg.image_size, cfg.image_size)
        return fallback, torch.tensor(0, dtype=torch.long)


# Standard transforms (224x224) used by MobileNetV2, VGG16, VGG19
train_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

eval_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

# InceptionV3 requires 299x299 input
inception_train_transform = transforms.Compose([
    transforms.Resize((cfg.inception_size, cfg.inception_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

inception_eval_transform = transforms.Compose([
    transforms.Resize((cfg.inception_size, cfg.inception_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])


def make_loader(frame, transform, batch_size, shuffle):
    return DataLoader(
        EmbryoDataset(frame, transform),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )

standard_loaders = {
    'train': make_loader(train_df, train_transform,       cfg.batch_size,           True),
    'val':   make_loader(val_df,   eval_transform,        cfg.batch_size,           False),
    'test':  make_loader(test_df,  eval_transform,        cfg.batch_size,           False),
}

inception_loaders = {
    'train': make_loader(train_df, inception_train_transform, cfg.inception_batch_size, True),
    'val':   make_loader(val_df,   inception_eval_transform,  cfg.inception_batch_size, False),
    'test':  make_loader(test_df,  inception_eval_transform,  cfg.inception_batch_size, False),
}

print('Standard train batches :', len(standard_loaders['train']))
print('Inception train batches:', len(inception_loaders['train']))

Standard train batches : 879
Inception train batches: 1758


## Loss Functions

### Cross Entropy (Baseline)

Standard weighted cross entropy with label smoothing.  
Class weights derived from inverse class frequency to address the inherent class imbalance across embryo stages.

### Smooth Ordinal Loss (Custom)

Embryo development stages follow a strict biological order.  
Misclassifying stage t3 as t4 is a much smaller error than misclassifying it as tHB.  
Standard cross entropy treats both mistakes identically.

The custom loss adds an exponential distance penalty on top of CE:

$$\mathcal{L} = \mathcal{L}_{CE} + \alpha \cdot \frac{1}{N} \sum_{i=1}^{N} \left( \exp\left(\frac{|\hat{y}_{soft,i} - y_i|}{\tau}\right) - 1 \right)$$

where the soft prediction is:

$$\hat{y}_{soft} = \sum_{k=0}^{C-1} k \cdot p_k$$

- The soft prediction is differentiable, enabling direct gradient-based optimisation.  
- The exponential term means distant stage errors are penalised far more severely than close ones.  
- tau controls how quickly the penalty grows with distance.  
- alpha balances CE against the ordinal penalty.

In [11]:
class SmoothOrdinalLoss(nn.Module):
    """
    Cross Entropy plus an exponential ordinal-distance penalty.

    Adapted from the MAPE concept: rather than penalising percentage error,
    a soft expected-class index is computed from the predicted probability
    distribution and compared against the true class index.

    Parameters
    ----------
    alpha : float
        Weight of the distance penalty relative to CE.
    tau : float
        Temperature for the exponential penalty; larger values make the
        penalty grow more slowly with distance.
    """

    def __init__(self, alpha: float = 0.3, tau: float = 3.0):
        super().__init__()
        self.alpha = alpha
        self.tau   = tau
        self.ce    = nn.CrossEntropyLoss()

    def forward(self, outputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce_loss = self.ce(outputs, targets)

        probs         = F.softmax(outputs, dim=1)
        class_indices = torch.arange(outputs.size(1), dtype=torch.float32, device=outputs.device)

        # Soft (differentiable) predicted class index
        y_pred_soft = torch.sum(probs * class_indices, dim=1)

        # Ordinal distance penalty
        distance = torch.abs(y_pred_soft - targets.float())
        penalty  = torch.exp(distance / self.tau) - 1.0

        return ce_loss + self.alpha * penalty.mean()


def make_loss(loss_name: str) -> nn.Module:
    if loss_name == 'ce':
        return nn.CrossEntropyLoss(
            weight=class_weights,
            label_smoothing=cfg.label_smoothing,
        ).to(DEVICE)

    if loss_name == 'smooth_ordinal':
        return SmoothOrdinalLoss(
            alpha=cfg.sol_alpha,
            tau=cfg.sol_tau,
        ).to(DEVICE)

    raise ValueError(f'Unknown loss: {loss_name}')

## Model Definitions

In [12]:
from torchvision.models import (
    mobilenet_v2, MobileNet_V2_Weights,
    vgg16, VGG16_Weights,
    vgg19, VGG19_Weights,
    inception_v3, Inception_V3_Weights
)
import torch.nn as nn

def get_model(name: str, num_classes: int) -> nn.Module:
    name = name.lower()

    if name == 'mobilenet':
        # ✅ load pretrained on CPU
        model = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)

        model.classifier = nn.Sequential(
            nn.Dropout(0.25),
            nn.Linear(model.last_channel, num_classes),
        )

    elif name == 'vgg16':
        model = vgg16(weights=VGG16_Weights.DEFAULT)

        model.classifier[6] = nn.Linear(
            model.classifier[6].in_features, num_classes
        )

    elif name == 'vgg19':
        model = vgg19(weights=VGG19_Weights.DEFAULT)

        model.classifier[6] = nn.Linear(
            model.classifier[6].in_features, num_classes
        )

    elif name == 'inception':
        model = inception_v3(
            weights=Inception_V3_Weights.DEFAULT,
            aux_logits=True
        )

        model.fc = nn.Linear(model.fc.in_features, num_classes)
        model.AuxLogits.fc = nn.Linear(
            model.AuxLogits.fc.in_features, num_classes
        )

    else:
        raise ValueError(f'Unknown model name: {name}')

    # ✅ move to GPU AFTER loading weights
    return model.to(DEVICE)

## Training and Evaluation Helpers

In [13]:
def unpack_outputs(outputs):
    """Handle both standard tensors and InceptionV3 named tuples."""
    if hasattr(outputs, 'logits'):
        return outputs.logits, getattr(outputs, 'aux_logits', None)
    if isinstance(outputs, tuple) and len(outputs) == 2:
        return outputs[0], outputs[1]
    return outputs, None


def train_one_epoch(model, loader, optimizer, loss_fn, model_name: str):
    model.train()
    running_loss = 0.0
    preds_all, targets_all = [], []

    progress = tqdm(loader, desc=f'{model_name} train', leave=False)
    for images, labels in progress:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        outputs              = model(images)
        main_logits, aux_log = unpack_outputs(outputs)

        loss = loss_fn(main_logits, labels)
        if model_name == 'inception' and aux_log is not None:
            loss = loss + cfg.aux_weight * loss_fn(aux_log, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()

        running_loss += loss.item()
        preds = torch.argmax(main_logits, dim=1)
        preds_all.extend(preds.detach().cpu().numpy())
        targets_all.extend(labels.detach().cpu().numpy())
        progress.set_postfix(loss=f'{loss.item():.4f}')

    epoch_loss = running_loss / len(loader)
    epoch_acc  = accuracy_score(targets_all, preds_all)
    epoch_f1   = f1_score(targets_all, preds_all, average='weighted', zero_division=0)
    return epoch_loss, epoch_acc, epoch_f1


@torch.no_grad()
def evaluate(model, loader, model_name: str):
    model.eval()
    preds_all, targets_all = [], []

    for images, labels in tqdm(loader, desc=f'{model_name} eval', leave=False):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        outputs     = model(images)
        main_logits, _ = unpack_outputs(outputs)
        preds       = torch.argmax(main_logits, dim=1)

        preds_all.extend(preds.detach().cpu().numpy())
        targets_all.extend(labels.detach().cpu().numpy())

    acc = accuracy_score(targets_all, preds_all)
    f1  = f1_score(targets_all, preds_all, average='weighted', zero_division=0)
    return acc, f1

## Training Runner

In [14]:
def run_experiment(model_name: str, loss_name: str):
    print(f'\nTraining {model_name.upper()} | loss = {loss_name}')
    print('-' * 60)

    loaders = inception_loaders if model_name == 'inception' else standard_loaders
    model   = get_model(model_name, num_classes)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        factor=0.5,
        patience=2
    )

    loss_fn = make_loss(loss_name)

    history     = []
    best_state  = None
    best_val_f1 = -1.0

    num_epochs = 10  # 🔥 increased epochs

    for epoch in range(1, num_epochs + 1):

        # ===== TRAIN =====
        train_loss, train_acc, train_f1 = train_one_epoch(
            model, loaders['train'], optimizer, loss_fn, model_name
        )

        # ===== VALIDATE =====
        val_acc, val_f1 = evaluate(model, loaders['val'], model_name)

        # ===== LR SCHEDULER =====
        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_f1)
        current_lr = optimizer.param_groups[0]['lr']

        if current_lr != old_lr:
            print(f'LR reduced: {old_lr:.2e} → {current_lr:.2e}')

        # ===== SAVE HISTORY =====
        history.append({
            'epoch':      epoch,
            'train_loss': train_loss,
            'train_acc':  train_acc,
            'train_f1':   train_f1,
            'val_acc':    val_acc,
            'val_f1':     val_f1,
            'lr':         current_lr,
        })

        # ===== PRINT =====
        print(
            f'Epoch {epoch:>2}/{num_epochs} | '
            f'Train Loss: {train_loss:.4f} | '
            f'Train Acc: {train_acc:.4f} | '
            f'Train F1: {train_f1:.4f} | '
            f'Val Acc: {val_acc:.4f} | '
            f'Val F1: {val_f1:.4f} | '
            f'LR: {current_lr:.2e}'
        )

        # ===== CHECKPOINT (BEST MODEL) =====
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state  = copy.deepcopy(model.state_dict())

            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_f1': val_f1,
            }, f'/kaggle/working/{model_name}_{loss_name}_best.pth')

    # ===== LOAD BEST MODEL =====
    if best_state is not None:
        model.load_state_dict(best_state)

    # ===== TEST =====
    test_acc, test_f1 = evaluate(model, loaders['test'], model_name)

    print(
        f'Best Val F1: {best_val_f1:.4f} | '
        f'Test Acc: {test_acc:.4f} | '
        f'Test F1: {test_f1:.4f}'
    )

    return pd.DataFrame(history), {
        'model':        model_name,
        'loss':         loss_name,
        'best_val_f1':  best_val_f1,
        'test_acc':     test_acc,
        'test_f1':      test_f1,
    }

## Experiment 1: Weighted Cross Entropy (Baseline)

All four models are trained using weighted cross entropy with label smoothing.

In [15]:
models_to_train = ['mobilenet', 'vgg16', 'vgg19', 'inception']

all_histories = []
all_results   = []

for model_name in models_to_train:
    hist_df, result = run_experiment(model_name, 'ce')
    hist_df['model'] = model_name
    hist_df['loss']  = 'ce'
    all_histories.append(hist_df)
    all_results.append(result)


Training MOBILENET | loss = ce
------------------------------------------------------------
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 116MB/s] 


mobilenet train:   0%|          | 0/879 [00:00<?, ?it/s]

mobilenet eval:   0%|          | 0/193 [00:00<?, ?it/s]

Epoch  1/10 | Train Loss: 2.3871 | Train Acc: 0.2313 | Train F1: 0.2425 | Val Acc: 0.2752 | Val F1: 0.2797 | LR: 1.00e-04


...

inception eval:   0%|          | 0/385 [00:00<?, ?it/s]

Epoch 10/10 | Train Loss: 2.1948 | Train Acc: 0.6101 | Train F1: 0.6200 | Val Acc: 0.3255 | Val F1: 0.3380 | LR: 5.00e-05


inception eval:   0%|          | 0/379 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7af5455ae2a0>Exception ignored in: 
<function _MultiProcessingDataLoaderIter.__del__ at 0x7af5455ae2a0>
Exception ignored in: Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
<function _MultiProcessingDataLoaderIter.__del__ at 0x7af5455ae2a0>  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__

    self._shutdown_workers()Traceback (most recent call last):
    
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
self._shutdown_workers()        
if w.is_alive():self._shutdown_workers()  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _s

Best Val F1: 0.3510 | Test Acc: 0.3320 | Test F1: 0.3446


## Experiment 2: Smooth Ordinal Loss (Custom)

All four models are retrained using the custom SmoothOrdinalLoss.

In [16]:
for model_name in models_to_train:
    hist_df, result = run_experiment(model_name, 'smooth_ordinal')
    hist_df['model'] = model_name
    hist_df['loss']  = 'smooth_ordinal'
    all_histories.append(hist_df)
    all_results.append(result)


Training MOBILENET | loss = smooth_ordinal
------------------------------------------------------------


mobilenet train:   0%|          | 0/879 [00:00<?, ?it/s]

mobilenet eval:   0%|          | 0/193 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7af5455ae2a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7af5455ae2a0>^
Traceback (most recent call last):

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
      File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    self._shutdown_workers()
assert self._parent_pid == os.getpid(), 'can only test a child process'  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

    if w.is_alive(): 
               ^ ^ ^^^^^^^^^^^^^^^^^^^
^  

Epoch  1/10 | Train Loss: 2.3879 | Train Acc: 0.3556 | Train F1: 0.2993 | Val Acc: 0.3653 | Val F1: 0.3222 | LR: 1.00e-04


...

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7af5455ae2a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7af5455ae2a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

inception eval:   0%|          | 0/385 [00:00<?, ?it/s]

Epoch 10/10 | Train Loss: 1.5419 | Train Acc: 0.6544 | Train F1: 0.6342 | Val Acc: 0.3808 | Val F1: 0.3584 | LR: 5.00e-05


inception eval:   0%|          | 0/379 [00:00<?, ?it/s]

Best Val F1: 0.3584 | Test Acc: 0.3856 | Test F1: 0.3621


## Final Results

In [21]:
history_table = pd.concat(all_histories, ignore_index=True)
results_table = (
    pd.DataFrame(all_results)
    .sort_values(['loss', 'test_f1'], ascending=[True, False])
    .reset_index(drop=True)
)

print('Full results table:')
display(results_table)

Full results table:


,model,loss,best_val_f1,test_acc,test_f1
0,inception,ce,0.351039,0.331958,0.344561
1,mobilenet,ce,0.314306,0.295642,0.308090
2,vgg16,ce,0.315458,0.292341,0.302328
3,vgg19,ce,0.304346,0.276659,0.291320
4,inception,smooth_ordinal,0.358432,0.385606,0.362084
5,mobilenet,smooth_ordinal,0.347630,0.379663,0.346552
6,vgg16,smooth_ordinal,0.361775,0.374711,0.335268
7,vgg19,smooth_ordinal,0.345060,0.380984,0.325560


In [18]:
print('Pivot: Test Accuracy and Test F1 by model and loss')
pivot = results_table.pivot(index='model', columns='loss', values=['test_acc', 'test_f1'])
display(pivot)

Pivot: Test Accuracy and Test F1 by model and loss


test_acc                  test_f1               
loss             ce smooth_ordinal        ce smooth_ordinal
model                                                      
inception  0.331958       0.385606  0.344561       0.362084
mobilenet  0.295642       0.379663  0.308090       0.346552
vgg16      0.292341       0.374711  0.302328       0.335268
vgg19      0.276659       0.380984  0.291320       0.325560

In [19]:
print('Best model per loss function:')
for loss_name in results_table['loss'].unique():
    subset = results_table[results_table['loss'] == loss_name]
    best   = subset.loc[subset['test_f1'].idxmax()]
    print(f'  {loss_name:20s} -> {best["model"]:12s} | Test F1: {best["test_f1"]:.4f} | Test Acc: {best["test_acc"]:.4f}')

Best model per loss function:
  ce                   -> inception    | Test F1: 0.3446 | Test Acc: 0.3320
  smooth_ordinal       -> inception    | Test F1: 0.3621 | Test Acc: 0.3856


# 1. Methodology
Embryo stages follow a natural sequence, the labels were treated as **ordinal categories** rather than independent classes. Additionally, class 15 was merged into class 14 due to its very low sample count and strong visual similarity, resulting in a total of 15 effective classes. To prevent data leakage, the dataset was split **embryo-wise**, ensuring that frames from the same embryo do not appear across training, validation, and test sets.

Four pretrained CNN architectures were evaluated:

- MobileNetV2  
- VGG16  
- VGG19  
- InceptionV3  

For each model, two training settings were considered:

- **Baseline:** Weighted CrossEntropy loss with label smoothing  
- **Custom loss:** Smooth Ordinal Loss  

Predicting a nearby stage is less severe than predicting a distant stage. Standard CrossEntropy does not capture this structure, as it only focuses on maximizing the probability of the correct class.

To address this, the Smooth Ordinal Loss extends CrossEntropy by adding a penalty based on the **distance between predicted and true stage indices**. The key idea is to compute a soft prediction:

$$
y_{\text{soft}} = \sum_{k=0}^{C-1} k \cdot p_k
$$

where $p_k$ is the softmax probability for class $k$. This provides a differentiable estimate of the predicted stage.

The final loss is defined as:

$$
L = L_{CE} + \alpha \left( e^{\frac{|y_{\text{soft}} - y|}{\tau}} - 1 \right)
$$

where:

- $L_{CE}$ is the weighted CrossEntropy loss  
- $|y_{\text{soft}} - y|$ measures ordinal distance  
- $\alpha$ controls the strength of the penalty  
- $\tau$ controls how quickly the penalty grows  

The loss is fully differentiable since it is composed of softmax, summation, and exponential functions, allowing it to be optimized using standard backpropagation.

---

# 2. Results and Analysis
Both CrossEntropy and Smooth Ordinal Loss were evaluated across all four models using test accuracy and weighted F1-score.

The results show a clear and consistent trend: **Smooth Ordinal Loss improves performance across all architectures**. In every model, both test accuracy and F1-score increased when compared to the standard CrossEntropy baseline.

The best overall performance was achieved by **InceptionV3 with Smooth Ordinal Loss**, which obtained the highest test accuracy (0.3856) and test F1-score (0.3621). This indicates that combining a strong architecture with an ordinal-aware loss function leads to better learning of embryo stage progression.

Significant improvements were also observed for the other models. For example, **MobileNetV2** showed a notable increase in both accuracy and F1-score when using Smooth Ordinal Loss, despite being a lightweight architecture. Similarly, **VGG16 and VGG19** both benefited from the ordinal loss, with consistent gains in performance over the CrossEntropy baseline. By penalizing predictions based on their distance from the true stage, the model learns more meaningful representations.

---

# 3. Conclusion

This work demonstrates that treating embryo stage classification as an **ordinal problem** leads to improved performance compared to standard classification approaches.

The proposed Smooth Ordinal Loss effectively incorporates stage-order information by penalizing predictions based on their distance from the true stage. This results in more meaningful learning, particularly for models capable of capturing complex visual patterns.

Among all configurations, **InceptionV3 combined with Smooth Ordinal Loss achieved the best performance**, making it the most suitable approach for this task.

In conclusion, the study highlights that aligning the loss function with the inherent structure of the problem is crucial. Incorporating ordinal relationships into deep learning models can significantly improve performance in tasks where class labels follow a natural progression, such as embryo development stages.